# Canonical Fixed DQA-Upcycled MoE v2

This experiment fixes the comparison problem from the previous run by reusing the strong canonical dense warmup and phase1 checkpoints. Phase2 then upcycles the dense model into anonymous DQA-conditioned backbone/neck adapters plus a lightweight anonymous head MoE.

- Warmup is not retrained.
- Phase1 is seeded from the canonical FedSTO-compatible checkpoint.
- Phase2 first trains only anonymous adapter/head-MoE capacity, then lightly opens the head near the end.
- Aggregation uses a small trust-region soft mixture so the strong dense model is not overwritten.


In [ ]:
from pathlib import Path

REPO = Path('/app/Object_Detection')
RUNNER = REPO / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region' / 'scripts' / 'run_dqa_anonymous_backbone_moe.py'
WORKSPACE = REPO / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region' / 'output' / '04_canonical_fixed_dqa_upcycled_moe_v2'
SOURCE_WORKSPACE = REPO / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region' / 'output' / '02_shared_routed_head_moe_12h'
CANONICAL_WARMUP = SOURCE_WORKSPACE / 'global_checkpoints' / 'round000_warmup.pt'
CANONICAL_PHASE1 = SOURCE_WORKSPACE / 'global_checkpoints' / 'phase1_round020_global.pt'
LOG = WORKSPACE / 'canonical_fixed_dqa_upcycled_moe_v2.log'
SUMMARY = WORKSPACE / 'anonymous_backbone_moe_round_summary.csv'

WORKSPACE.mkdir(parents=True, exist_ok=True)
assert RUNNER.exists(), RUNNER
assert CANONICAL_WARMUP.exists(), CANONICAL_WARMUP
assert CANONICAL_PHASE1.exists(), CANONICAL_PHASE1
RUNNER, WORKSPACE, CANONICAL_WARMUP, CANONICAL_PHASE1, LOG, SUMMARY


In [ ]:
cmd = [
    'python', str(RUNNER),
    '--workspace-root', str(WORKSPACE),
    '--protocol-suffix', 'canonical_fixed_upcycled_head_moe_v1',
    '--warmup-checkpoint', str(CANONICAL_WARMUP),
    '--phase1-checkpoint', str(CANONICAL_PHASE1),
    '--warmup-epochs', '50',
    '--phase1-rounds', '20',
    '--phase2-rounds', '20',
    '--moe-start-phase', '2',
    '--phase2-train-scope', 'backbone_adapter_moe_head_moe',
    '--phase2-late-train-scope', 'backbone_adapter_moe_head',
    '--phase2-head-unfreeze-after-round', '15',
    '--orthogonal-weight', '0.0001',
    '--batch-size', '128',
    '--phase2-batch-size', '64',
    '--phase2-server-lr0', '0.001',
    '--workers', '32',
    '--gpus', '2',
    '--master-port', '29561',
    '--moe-num-experts', '4',
    '--moe-top-k', '2',
    '--moe-temperature', '1.0',
    '--moe-scale', '0.25',
    '--moe-shared-scale', '1.0',
    '--moe-adapter-ratio', '0.125',
    '--moe-levels', 'c3,c4,c5',
    '--moe-kernels', '3,5,7',
    '--moe-context-dim', '8',
    '--moe-quality-dim', '4',
    '--moe-router-noise-std', '0.005',
    '--moe-balance-weight', '0.03',
    '--moe-entropy-weight', '0.002',
    '--moe-z-loss-weight', '0.0001',
    '--moe-diversity-weight', '0.002',
    '--enable-head-moe',
    '--head-moe-start-phase', '2',
    '--head-moe-num-experts', '4',
    '--head-moe-top-k', '2',
    '--head-moe-temperature', '1.0',
    '--head-moe-scale', '0.25',
    '--head-moe-balance-weight', '0.01',
    '--head-moe-entropy-weight', '0.001',
    '--enable-neck-moe',
    '--neck-moe-start-phase', '2',
    '--neck-moe-num-experts', '4',
    '--neck-moe-top-k', '2',
    '--neck-moe-temperature', '1.0',
    '--neck-moe-scale', '0.15',
    '--neck-moe-shared-scale', '1.0',
    '--neck-moe-adapter-ratio', '0.0625',
    '--neck-moe-levels', 'p3,p4,p5',
    '--neck-moe-kernels', '3,5',
    '--neck-moe-router-noise-std', '0.005',
    '--neck-moe-balance-weight', '0.02',
    '--neck-moe-entropy-weight', '0.001',
    '--neck-moe-z-loss-weight', '0.0001',
    '--neck-moe-diversity-weight', '0.001',
    '--phase2-aggregate-lambda', '0.10',
    '--phase2-max-relative-update', '0.005',
    '--phase2-max-absolute-update', '0.0',
    '--phase2-aggregate-scope', 'adapter_head',
    '--router-diagnostic-split', 'cloudy',
    '--router-diagnostic-images', '16',
    '--router-diagnostic-batch-size', '4',
    '--run-final-eval',
    '--final-eval-splits', 'cloudy,overcast,rainy,snowy,total',
    '--val-batch-size', '32',
    '--discord',
]

print(' '.join(cmd))


In [ ]:
import subprocess
from datetime import datetime, timezone

with LOG.open('a', encoding='utf-8') as f:
    f.write(f'\n\n===== started {datetime.now(timezone.utc).isoformat()} =====\n')
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=f, stderr=subprocess.STDOUT, text=True)

print(f'pid={proc.pid}')
print(f'log={LOG}')


In [ ]:
import pandas as pd

if SUMMARY.exists():
    display(pd.read_csv(SUMMARY).tail(30))
else:
    print('summary is not created yet')

router_csv = WORKSPACE / 'anonymous_backbone_moe_router_diagnostics.csv'
if router_csv.exists():
    display(pd.read_csv(router_csv).tail(30))

if LOG.exists():
    print(''.join(LOG.read_text(encoding='utf-8', errors='replace').splitlines(True)[-120:]))
